# ➕ Add New Idiom & Find Synonyms

Notebook này cho phép bạn:
1. **Nhập tên câu thành ngữ mới** (tiếng Việt hoặc tiếng Anh)
2. **Chọn các tiêu chí** từ Bể tiêu chí chung có sẵn
3. **Chạy HermiT Reasoner** để tự động tìm các câu đồng nghĩa

> ⚠️ File `v-idiomV5.rdf` gốc **KHÔNG bị thay đổi** — mọi thứ chạy trong bộ nhớ RAM.

In [ ]:
# ============================================================
# CELL 1: TẢI ONTOLOGY + TRÍCH XUẤT BỂ TIÊU CHÍ
# ============================================================
from owlready2 import *

BASE_IRI = "http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv2#"

temp_world = World()
onto_r = temp_world.get_ontology("../../ontology_protege/v-idiomV5.rdf").load()

# Lấy các class
Khung_cls = temp_world[BASE_IRI + "Khung_Bản_Thể_học"]
VN_cls    = temp_world[BASE_IRI + "Vietnamese_idiom"]
EN_cls    = temp_world[BASE_IRI + "English_Idiom"]
BC_cls    = temp_world[BASE_IRI + "Bối_cảnh"]
HD_cls    = temp_world[BASE_IRI + "Hành_động"]
KQ_cls    = temp_world[BASE_IRI + "Kết_quả"]
MD_cls    = temp_world[BASE_IRI + "Mục_đích"]

# Trích xuất danh sách các tiêu chí đã có trong ontology
NONE_OPT = "(Không chọn)"
bc_options = [NONE_OPT] + sorted([i.name for i in BC_cls.instances()])
hd_options = [NONE_OPT] + sorted([i.name for i in HD_cls.instances()])
kq_options = [NONE_OPT] + sorted([i.name for i in KQ_cls.instances()])
md_options = [NONE_OPT] + sorted([i.name for i in MD_cls.instances()])

print(f"✅ Tải xong ontology: {onto_r.base_iri}")
print(f"   Bối cảnh   : {len(bc_options)-1} loại")
print(f"   Hành động  : {len(hd_options)-1} loại")
print(f"   Kết quả    : {len(kq_options)-1} loại")
print(f"   Mục đích   : {len(md_options)-1} loại")

In [ ]:
# ============================================================
# CELL 2: GIAO DIỆN NHẬP LIỆU TƯƠNG TÁC
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# --- Widget nhập tên thành ngữ ---
w_name = widgets.Text(
    placeholder='Ví dụ: Liều_lĩnh_như_con_thiêu_thân',
    description='Tên (dùng _):',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '130px'}
)

# --- Widget chọn ngôn ngữ ---
w_lang = widgets.RadioButtons(
    options=[('🇻🇳 Tiếng Việt', 'vn'), ('🇬🇧 Tiếng Anh', 'en')],
    description='Ngôn ngữ:',
    style={'description_width': '130px'}
)

# --- Dropdown 4 tiêu chí ---
w_bc = widgets.Dropdown(
    options=bc_options, description='Bối cảnh:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '130px'}
)
w_hd = widgets.Dropdown(
    options=hd_options, description='Hành động:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '130px'}
)
w_kq = widgets.Dropdown(
    options=kq_options, description='Kết quả:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '130px'}
)
w_md = widgets.Dropdown(
    options=md_options, description='Mục đích:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '130px'}
)

# --- Nút chạy ---
btn_run = widgets.Button(
    description='🔍 Thêm & Tìm đồng nghĩa',
    button_style='success',
    layout=widgets.Layout(width='250px', height='40px')
)
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        name = w_name.value.strip().replace(' ', '_')
        lang = w_lang.value
        bc   = w_bc.value if w_bc.value != NONE_OPT else None
        hd   = w_hd.value if w_hd.value != NONE_OPT else None
        kq   = w_kq.value if w_kq.value != NONE_OPT else None
        md   = w_md.value if w_md.value != NONE_OPT else None

        if not name:
            print("❌ Vui lòng nhập tên thành ngữ!")
            return
        if not any([bc, hd, kq, md]):
            print("❌ Vui lòng chọn ít nhất 1 tiêu chí!")
            return

        print(f"📝 Đang thêm: [{lang.upper()}] {name}")
        print(f"   Bối cảnh : {bc or '—'}")
        print(f"   Hành động: {hd or '—'}")
        print(f"   Kết quả  : {kq or '—'}")
        print(f"   Mục đích : {md or '—'}")
        print()

        # --- Thêm vào ontology (tạm thời trong RAM) ---
        with onto_r:
            # Tìm hoặc tạo Khung phù hợp
            signature = (bc or "", hd or "", kq or "", md or "")
            matched_frame = None
            for khung in Khung_cls.instances():
                bc_match = (bc is None) or (khung.Có_bối_cảnh and khung.Có_bối_cảnh[0].name == bc)
                hd_match = (hd is None) or (khung.Có_hành_động and khung.Có_hành_động[0].name == hd)
                kq_match = (kq is None) or (khung.Có_kết_quả and khung.Có_kết_quả[0].name == kq)
                md_match = (md is None) or (khung.Có_mục_đích and khung.Có_mục_đích[0].name == md)
                # Khớp chính xác 4 tiêu chí (kể cả None)
                bc_exact = (bc is None and not khung.Có_bối_cảnh) or bc_match
                hd_exact = (hd is None and not khung.Có_hành_động) or hd_match
                kq_exact = (kq is None and not khung.Có_kết_quả) or kq_match
                md_exact = (md is None and not khung.Có_mục_đích) or md_match
                if bc_exact and hd_exact and kq_exact and md_exact:
                    matched_frame = khung
                    break

            if matched_frame:
                print(f"✅ Tìm thấy Khung hiện có: [{matched_frame.name}] — câu mới sẽ được ghép vào đây")
            else:
                # Tạo Khung mới
                new_id = len(list(Khung_cls.instances())) + 4
                matched_frame = Khung_cls(f"Khung{new_id}")
                if bc: matched_frame.Có_bối_cảnh  = [BC_cls(bc)]
                if hd: matched_frame.Có_hành_động = [HD_cls(hd)]
                if kq: matched_frame.Có_kết_quả   = [KQ_cls(kq)]
                if md: matched_frame.Có_mục_đích  = [MD_cls(md)]
                print(f"🆕 Tạo Khung mới: [{matched_frame.name}]")

            # Tạo thực thể thành ngữ
            if lang == 'vn':
                new_idiom = VN_cls(name)
            else:
                new_idiom = EN_cls(name)
            new_idiom.Có_khung = [matched_frame]

        print()
        print("🟣 Đang chạy HermiT Reasoner...")
        t0 = time.time()
        with onto_r:
            sync_reasoner(x=temp_world)
        elapsed = time.time() - t0
        print(f"✅ HermiT chạy xong! Thời gian: {elapsed:.3f} giây")
        print()

        # --- Hiển thị các câu đồng nghĩa ---
        synonyms_vn = []
        synonyms_en = []
        for vi_id in VN_cls.instances():
            if vi_id.name != name and matched_frame in (vi_id.Có_khung or []):
                synonyms_vn.append(vi_id.name.replace('_', ' '))
        for en_id in EN_cls.instances():
            if en_id.name != name and matched_frame in (en_id.Có_khung or []):
                synonyms_en.append(en_id.name.replace('_', ' '))

        print("=" * 55)
        print(f"  KẾT QUẢ: Các câu đồng nghĩa với '{name.replace('_',' ')}':")
        print("=" * 55)
        if not synonyms_vn and not synonyms_en:
            print("  🔸 Chưa có câu đồng nghĩa nào trong hệ thống.")
        for vn in synonyms_vn:
            print(f"  🇻🇳  {vn}")
        for en in synonyms_en:
            print(f"  🇬🇧  {en}")
        print("=" * 55)

btn_run.on_click(on_click)

display(
    widgets.VBox([
        widgets.HTML("<h3>📝 Nhập thông tin thành ngữ mới</h3>"),
        w_name, w_lang,
        widgets.HTML("<hr><b>Chọn tiêu chí từ Bể tiêu chí chung:</b>"),
        w_bc, w_hd, w_kq, w_md,
        widgets.HTML("<hr>"),
        btn_run,
        out
    ])
)